# 02 - Preprocessing, Feature Engineering, and Operating-Condition Analysis

**Project:** Naval Propulsion Clustering  
**Academic Title:** *Unsupervised Discovery of Operating and Performance Degradation Profiles in Naval Gas Turbine Propulsion Systems*  
**Phase:** Phase 2 — Preprocessing, Scaling, and Feature Engineering  

This notebook documents the experimental evaluation of data preprocessing choices for Phase 3 clustering.
All analytical logic, transformers, and diagnostics are imported from `naval_propulsion`.

In [ ]:
import matplotlib
matplotlib.use('Agg')  # Headless backend to prevent GUI event loop blocking
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from naval_propulsion.data.loader import load_naval_dataset
from naval_propulsion.features.correlation import compute_correlations, REPRESENTATION_A_FEATURES, REPRESENTATION_B_FEATURES
from naval_propulsion.features.pca_analysis import analyze_pca
from naval_propulsion.features.registry import CANDIDATE_REPRESENTATIONS, build_representation_data
from naval_propulsion.features.screening import screen_features
from naval_propulsion.preprocessing.outliers import analyze_outliers
from naval_propulsion.preprocessing.regime import analyze_operating_regimes
from naval_propulsion.preprocessing.scalers import compare_scalers
from naval_propulsion.visualization.plots import configure_plot_style

configure_plot_style()
print("Phase 2 environment and naval_propulsion package initialized.")

## 1. Feature Screening: Zero-Variance and Duplicate Detection

We evaluate which features must be removed due to zero variance or exact duplication, and distinguish mandatory removal from candidate retention.

In [ ]:
container = load_naval_dataset()
full_df = pd.concat([container.features, container.targets], axis=1)

screening_df = screen_features(full_df)
screening_df

## 2. Correlation Analysis: Representation A vs Representation B

- **Representation A (Minimal-Cleaning):** Retains all 13 valid non-constant, non-duplicate sensors.
- **Representation B (Reduced-Correlation):** Prunes redundant collinear features (|r| >= 0.985) across distinct thermodynamic subsystems.

In [ ]:
print("Representation A feature count:", len(REPRESENTATION_A_FEATURES))
print("Representation A features:", REPRESENTATION_A_FEATURES)
print("\nRepresentation B feature count:", len(REPRESENTATION_B_FEATURES))
print("Representation B features:", REPRESENTATION_B_FEATURES)

corrs_b = compute_correlations(full_df, features=REPRESENTATION_B_FEATURES)
print("\nMax off-diagonal Pearson r in Representation B:", 
      corrs_b['pearson'].replace(1.0, 0.0).abs().max().max())

## 3. Scaler Comparison: StandardScaler vs RobustScaler vs MinMaxScaler

We compare transformed moments, quantiles, and extreme value compression across standard, robust, and min-max scalers.

In [ ]:
scaler_comparison = compare_scalers(full_df, features=REPRESENTATION_A_FEATURES)
for stype, info in scaler_comparison.items():
    print(f"Scaler: {stype:10s} | Overall Range: [{info['overall_min']:.2f}, {info['overall_max']:.2f}] | Overall Std: {info['overall_std']:.2f}")

## 4. Outlier Investigation & Classification

Univariate (IQR and robust z-score) and multivariate (Mahalanobis distance) diagnostics are computed to evaluate whether extremes represent physical operating boundaries.

In [ ]:
outliers = analyze_outliers(full_df, features=REPRESENTATION_A_FEATURES)
print("Classification breakdown:", outliers['classification_counts'])
print("Physical interpretation:", outliers['physical_interpretation'])

## 5. Operating Condition Decomposition: Between-Speed vs Within-Speed Variance

We measure the proportion of total feature variance explained by commanded ship speed ($v$).

In [ ]:
regime_report = analyze_operating_regimes(full_df, regime_col="v")
print(f"Mean variance explained across all sensors: {regime_report['mean_variance_explained_by_speed']:.2f}%")
for feat, metrics in regime_report['feature_metrics'].items():
    print(f"  {feat:6s}: {metrics['percent_variance_explained_by_speed']:.2f}% explained by speed")

## 6. PCA Explained Variance & Dominance Diagnosis

We investigate why PC1 dominates the feature space.

In [ ]:
pca_rep = analyze_pca(full_df, features=REPRESENTATION_A_FEATURES)
print(f"PC1 Explained Variance: {pca_rep['pc1_explained_variance_percent']:.2f}%")
print(f"PC1 + PC2 Cumulative:  {pca_rep['pc1_plus_pc2_explained_variance_percent']:.2f}%")
print("\nScientific interpretation:", pca_rep['scientific_interpretation'])

## 7. Feature Representation Registry Summary

Summary of the 5 formal candidate representations prepared for Phase 3 clustering experiments.

In [ ]:
reg_df = pd.DataFrame([
    {
        "ID": spec.representation_id,
        "Features Count": len(spec.feature_names),
        "Scaler": spec.scaler_type,
        "Normalization": spec.normalization,
        "Status": spec.status,
        "Intended Use": spec.intended_use
    }
    for spec in CANDIDATE_REPRESENTATIONS.values()
])
reg_df